# Quick Start

How a model is written and solved with EconPDEs.jl, using the panel's own code: long-run risk in the manner of [Bansal and Yaron (2004)](https://doi.org/10.1111/j.1540-6261.2004.00670.x), with constant volatility and one state, expected consumption growth `mu`. The other notebooks of this panel adapt the package's own examples.

In [ ]:
Pollis.packages("EconPDEs", "Plots")

## Writing the model

A model is a callable struct. It receives `state`, one point of the grid, and `u`, the unknown with its finite differences at that point. Names are put together from the unknown and the state: for an unknown `p` and a state `mu`, they are `p`, `pmu_up`, `pmu_down` and `pmumu`.

The first derivative is taken on the side the drift points to (upwinding). The function returns the time derivative of the unknown, `pt`, which is zero at the solution. A second NamedTuple, which is optional, holds values to keep on the grid.

In [ ]:
using EconPDEs

# Bansal-Yaron (2004) long-run risk with constant volatility, in continuous time and annual units:
#   dC/C = mu dt + sigma dZc,   d mu = kappa (mubar - mu) dt + nu sigma dZmu
# The unknown is the wealth-consumption ratio p(mu) of the consumption claim (two-state version: EconPDEs examples/asset_pricing/bansal_yaron.jl)
Base.@kwdef struct LongRunRisk
    mubar::Float64; sigma::Float64; kappa::Float64; nu::Float64
    rho::Float64; gamma::Float64; psi::Float64
end

function (m::LongRunRisk)(state::NamedTuple, u::NamedTuple)
    (; mubar, sigma, kappa, nu, rho, gamma, psi) = m
    (; mu) = state
    (; p, pmu_up, pmu_down, pmumu) = u
    drift = kappa * (mubar - mu)  # drift of expected growth
    vol   = nu * sigma  # volatility of expected growth
    pmu   = drift >= 0 ? pmu_up : pmu_down  # upwind on the sign of the drift
    sigma_p = pmu / p * vol  # volatility of p from growth news
    mu_p    = pmu / p * drift + 0.5 * pmumu / p * vol^2  # drift of p, divided by p
    price_c  = gamma * sigma  # market price of short-run consumption risk
    price_mu = (gamma * psi - 1) / (psi - 1) * sigma_p  # market price of long-run risk (zero under power utility, gamma psi = 1)
    r = rho + mu / psi - (1 + 1 / psi) / 2 * gamma * sigma^2 - (gamma * psi - 1) / (2 * (psi - 1)) * sigma_p^2  # risk-free rate
    excess = price_c * sigma + price_mu * sigma_p  # expected excess return on the consumption claim
    vol_R  = sqrt(sigma^2 + sigma_p^2)  # return volatility
    pt = -p * (1 / p + mu + mu_p - r - excess)  # zero at the solution: expected return = r + excess
    return (; pt), (; r, excess, vol_R, sharpe = excess / vol_R, msr = sqrt(price_c^2 + price_mu^2))
end

## Solving it

`pdesolve` takes the model, the grid and a starting guess. `result.solution` holds the unknown, and `result.saved` holds what the function kept. The grid spans three stationary standard deviations of `mu` either side of its mean.

In [ ]:
m  = LongRunRisk(mubar = 0.018, sigma = 0.027, kappa = 0.252, nu = 0.528, rho = 0.024, gamma = 7.5, psi = 1.5)
sd = m.nu * m.sigma / sqrt(2 * m.kappa)  # stationary sd of expected growth (OU)
mus = range(m.mubar - 3sd, m.mubar + 3sd; length = 100)  # grid over expected growth

result = pdesolve(m, (; mu = mus), (; p = ones(100)))
p = result.solution.p  # wealth-consumption ratio on the grid, in years of consumption
s = result.saved  # r, excess, vol_R, sharpe and msr on the grid
w = exp.(-0.5 .* ((mus .- m.mubar) ./ sd) .^ 2); w = w ./ sum(w)  # stationary (normal) density of mu on the grid, for unconditional moments

## Checking it

`result.converged` and `result.residual_norm` say whether the equations hold on the grid. Read them before anything else. The means use the stationary density of `mu`.

In [ ]:
println("Converged: ", result.converged)
println("Mean wealth-consumption ratio: ", round(sum(w .* p), digits=1), " years")
println("Mean excess return on the consumption claim: ", round(100 * sum(w .* s.excess), digits=2), "% a year")
println("Mean risk-free rate: ", round(100 * sum(w .* s.r), digits=2), "% a year")
println("Residual norm: ", result.residual_norm)

## Plotting it

The wealth-consumption ratio rises with expected growth when the elasticity of intertemporal substitution `psi` is above one. The expected excess return on the consumption claim is close to 1.1% over most of the grid, since volatility is constant in this version.

It falls at both ends because of where the grid stops: the derivative of `p` goes to zero at the edges, so `sigma_p` vanishes and only the short-run part `gamma * sigma^2`, about 0.55%, remains. That is an effect of the grid, not a result of the model.

In [ ]:
using Plots
plot(plot(mus, p; xlabel = "expected growth mu", ylabel = "wealth-consumption ratio p", legend = false),
     plot(mus, 100 .* s.excess; xlabel = "expected growth mu", ylabel = "expected excess return, %", legend = false);
     layout = (1, 2), size = (820, 340), left_margin = 6Plots.mm, bottom_margin = 6Plots.mm)

## Changing a parameter

With `gamma * psi = 1`, recursive preferences reduce to power utility and the price of long-run risk `price_mu` is zero. Solving the same model there shows how much of the excess return long-run risk accounts for.

In [ ]:
power = LongRunRisk(mubar = m.mubar, sigma = m.sigma, kappa = m.kappa, nu = m.nu, rho = m.rho, gamma = m.gamma, psi = 1 / m.gamma)
result_power = pdesolve(power, (; mu = mus), (; p = ones(length(mus))))
println("Converged: ", result_power.converged)
println("Mean excess return, recursive preferences: ", round(100 * sum(w .* s.excess), digits=2), "% a year")
println("Mean excess return, power utility: ", round(100 * sum(w .* result_power.saved.excess), digits=2), "% a year")

## References

Bansal, R. and Yaron, A. (2004). Risks for the long run: a potential resolution of asset pricing puzzles. *The Journal of Finance* 59(4), 1481–1509. [doi:10.1111/j.1540-6261.2004.00670.x](https://doi.org/10.1111/j.1540-6261.2004.00670.x)